In [1]:
import os

In [2]:
%pwd

'/home/user/Documents/Kidney_Disease Using Deep Learning/research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'/home/user/Documents/Kidney_Disease Using Deep Learning'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_batch_size: int
    params_epochs: int
    params_is_argumentation: bool
    training_data: Path
    params_learning_rate: float

In [6]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml,create_directories
import tensorflow as tf

I0000 00:00:1777228643.920250   59467 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777228643.921201   59467 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1777228644.014641   59467 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777228646.957906   59467 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path=CONFIG_FILE_PATH,
        params_file_path=PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)

        create_directories([self.config.artifacts_root])


    def get_training_config(self) -> TrainingConfig:

        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        data_ingestion = self.config.data_ingestion

        training_data = os.path.join(
            data_ingestion.unzip_dir,
            "kidney-ct-scan-image"
        )

        create_directories([Path(training.root_dir)])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),

            params_image_size=self.params.IMAGE_SIZE,
            training_data=Path(training_data),

            params_batch_size=self.params.BATCH_SIZE,
            params_epochs=self.params.EPOCHS,
            params_is_argumentation=self.params.AUGMENTATION,
            params_learning_rate=self.params.LEARNING_RATE
        )

        return training_config

In [8]:
import os
import urllib.request as request
from zipfile import ZipFile
import time
import tensorflow as tf

In [9]:
class Training: 
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path,
            compile=False 
        )
        self.model.compile(
        optimizer=tf.keras.optimizers.SGD(
            learning_rate=self.config.params_learning_rate
        ),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale=1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_argumentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        # 🔥 FIX 1: moved outside else (important bug fix)
        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",   # 🔥 FIX 2: typo corrected
            shuffle=True,
            **dataflow_kwargs
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def train(self):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )

In [10]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
    
except Exception as e:
    raise e

2026-04-27 00:07:29,529 - INFO - cnnClassifier
2026-04-27 00:07:29,535 - INFO - cnnClassifier
2026-04-27 00:07:29,537 - INFO - cnnClassifier
2026-04-27 00:07:29,538 - INFO - cnnClassifier


E0000 00:00:1777228649.586405   59467 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Found 93 images belonging to 2 classes.
Found 372 images belonging to 2 classes.
Epoch 1/15


I0000 00:00:1777228651.068413   59467 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


23/23 ━━━━━━━━━━━━━━━━━━━━ 100s 4s/step - accuracy: 0.5646 - loss: 0.6971 - val_accuracy: 0.9625 - val_loss: 0.5420
Epoch 2/15
 1/23 ━━━━━━━━━━━━━━━━━━━━ 1:18 4s/step - accuracy: 0.5625 - loss: 0.6373

/home/user/Documents/Kidney_Disease Using Deep Learning/venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


23/23 ━━━━━━━━━━━━━━━━━━━━ 21s 792ms/step - accuracy: 0.5625 - loss: 0.6373 - val_accuracy: 0.9000 - val_loss: 0.5576
Epoch 3/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 110s 5s/step - accuracy: 0.6657 - loss: 0.6065 - val_accuracy: 0.9500 - val_loss: 0.5337
Epoch 4/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 25s 931ms/step - accuracy: 0.9375 - loss: 0.4783 - val_accuracy: 0.9625 - val_loss: 0.5315
Epoch 5/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 103s 4s/step - accuracy: 0.7388 - loss: 0.5513 - val_accuracy: 0.9000 - val_loss: 0.5330
Epoch 6/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 22s 817ms/step - accuracy: 0.6875 - loss: 0.5537 - val_accuracy: 0.9250 - val_loss: 0.5305
Epoch 7/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 101s 4s/step - accuracy: 0.7612 - loss: 0.5137 - val_accuracy: 0.8125 - val_loss: 0.5485
Epoch 8/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 19s 685ms/step - accuracy: 0.8125 - loss: 0.4500 - val_accuracy: 0.6125 - val_loss: 0.5891
Epoch 9/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 98s 4s/step - accuracy: 0.7978 - loss: 0.4896 - val_accuracy: 0.8625 - val_l